In [ ]:
import os
import subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from tqdm import tqdm
import json

print("📦 Importing libraries...")

# Set up nnU-Net environment variables
# These paths should point to your data directories
nnunet_raw = "/path/to/nnUNet_raw"  # Update this path
nnunet_preprocessed = "/path/to/nnUNet_preprocessed"  # Update this path
nnunet_results = "/path/to/nnUNet_results"  # Update this path

# Set environment variables
os.environ['nnUNet_raw'] = nnunet_raw
os.environ['nnUNet_preprocessed'] = nnunet_preprocessed
os.environ['nnUNet_results'] = nnunet_results

# Create directories if they don't exist
for path in [nnunet_raw, nnunet_preprocessed, nnunet_results]:
    Path(path).mkdir(parents=True, exist_ok=True)

print(f"✅ Environment setup complete!")
print(f"📁 nnUNet_raw: {nnunet_raw}")
print(f"📁 nnUNet_preprocessed: {nnunet_preprocessed}")
print(f"📁 nnUNet_results: {nnunet_results}")


In [ ]:
# Specify your dataset ID and name
DATASET_ID = 001  # Change this to your dataset ID
DATASET_NAME = f"Dataset{DATASET_ID:03d}_YourDataset"  # Change this to your dataset name

# Path to your dataset
dataset_path = Path(nnunet_raw) / DATASET_NAME
print(f"🔍 Looking for dataset: {dataset_path}")

# Check if dataset exists
if not dataset_path.exists():
    print(f"❌ Dataset not found at {dataset_path}")
    print("💡 Make sure your dataset is properly placed in nnUNet_raw folder")
    print("💡 Expected structure:")
    print("   nnUNet_raw/")
    print("   └── Dataset001_YourDataset/")
    print("       ├── dataset.json")
    print("       ├── imagesTr/")
    print("       ├── labelsTr/")
    print("       └── imagesTs/ (optional)")
else:
    print(f"✅ Dataset found!")

    # List dataset contents
    print(f"\n📁 Dataset contents:")
    for item in sorted(dataset_path.iterdir()):
        if item.is_dir():
            count = len(list(item.glob("*")))
            print(f"   📂 {item.name}/ ({count} files)")
        else:
            print(f"   📄 {item.name}")

    # Check dataset.json if it exists
    dataset_json_path = dataset_path / "dataset.json"
    if dataset_json_path.exists():
        with open(dataset_json_path, 'r') as f:
            dataset_info = json.load(f)

        print(f"\n📋 Dataset Information:")
        print(f"   Name: {dataset_info.get('name', 'N/A')}")
        print(f"   Description: {dataset_info.get('description', 'N/A')}")
        print(f"   Modalities: {dataset_info.get('modality', 'N/A')}")
        print(f"   Labels: {dataset_info.get('labels', 'N/A')}")
        print(f"   Training cases: {dataset_info.get('numTraining', 'N/A')}")
        print(f"   Test cases: {dataset_info.get('numTest', 'N/A')}")
    else:
        print(f"⚠️ dataset.json not found - this is required for nnU-Net")


In [ ]:
def run_nnunet_preprocessing(dataset_id, verify_dataset=True):
    """
    Run nnU-Net preprocessing for the specified dataset.

    Args:
        dataset_id (int): Dataset ID
        verify_dataset (bool): Whether to verify dataset integrity
    """

    # Check if preprocessing already exists
    preprocessed_path = Path(nnunet_preprocessed) / f"Dataset{dataset_id:03d}_YourDataset"
    if preprocessed_path.exists():
        print(f"✅ Preprocessing already exists at {preprocessed_path}")
        return True

    print(f"🔄 Starting preprocessing for Dataset{dataset_id:03d}...")

    # Build the preprocessing command
    cmd = [
        "nnUNetv2_plan_and_preprocess",
        "-d", str(dataset_id),
        "--verify_dataset_integrity" if verify_dataset else ""
    ]

    # Remove empty strings
    cmd = [c for c in cmd if c]

    print(f"🚀 Running command: {' '.join(cmd)}")

    try:
        # Run preprocessing
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=3600  # 1 hour timeout
        )

        if result.returncode == 0:
            print(f"✅ Preprocessing completed successfully!")
            print(f"📁 Preprocessed data saved to: {preprocessed_path}")
            return True
        else:
            print(f"❌ Preprocessing failed with return code: {result.returncode}")
            print(f"Error output: {result.stderr}")
            return False

    except subprocess.TimeoutExpired:
        print("⏰ Preprocessing timed out after 1 hour")
        return False
    except Exception as e:
        print(f"❌ Error during preprocessing: {e}")
        return False

# Run preprocessing
if dataset_path.exists():
    preprocessing_success = run_nnunet_preprocessing(DATASET_ID)
else:
    print("❌ Cannot run preprocessing - dataset not found")
    preprocessing_success = False


In [ ]:
def run_nnunet_training(dataset_id, configuration="2d", fold=0, trainer="nnUNetTrainer"):
    """
    Run nnU-Net training for the specified configuration.

    Args:
        dataset_id (int): Dataset ID
        configuration (str): Model configuration (2d, 3d_fullres, 3d_lowres)
        fold (int): Cross-validation fold (0-4)
        trainer (str): Trainer class name
    """

    # Check if model already exists
    model_path = Path(nnunet_results) / f"Dataset{dataset_id:03d}_YourDataset" / trainer / f"nnUNetPlans__{configuration}"
    if model_path.exists():
        print(f"✅ Model already exists at {model_path}")
        return True

    print(f"🏋️ Starting training for Dataset{dataset_id:03d}, {configuration}, fold {fold}...")

    # Build training command
    cmd = [
        "nnUNetv2_train",
        str(dataset_id),
        configuration,
        str(fold),
        "-tr", trainer
    ]

    print(f"🚀 Running command: {' '.join(cmd)}")
    print("⏰ This will take several hours depending on your dataset size and GPU...")

    try:
        # For demo purposes, we'll show the command but not actually run it
        # Uncomment the lines below to actually run training

        print("💡 Training command ready to execute.")
        print("💡 To run actual training, uncomment the subprocess.run() lines below.")

        # result = subprocess.run(
        #     cmd,
        #     capture_output=False,  # Show real-time output
        #     text=True
        # )
        #
        # if result.returncode == 0:
        #     print(f"✅ Training completed successfully!")
        #     return True
        # else:
        #     print(f"❌ Training failed with return code: {result.returncode}")
        #     return False

        return True  # For demo purposes

    except Exception as e:
        print(f"❌ Error during training: {e}")
        return False

# Option 1: Train from scratch (uncomment to run actual training)
if preprocessing_success:
    print("📋 Training options:")
    print("   1. 2D U-Net (fastest, good for 2D images)")
    print("   2. 3D full resolution (best for small 3D volumes)")
    print("   3. 3D cascade (best for large 3D volumes)")

    training_success = run_nnunet_training(DATASET_ID, configuration="2d", fold=0)
else:
    print("❌ Cannot train - preprocessing not completed")
    training_success = False

# Option 2: Download pretrained model (if available)
print("\n💡 Alternative: You can download pretrained models using:")
print("   nnUNetv2_download_pretrained_model_by_url <URL>")
print("   or browse available models at:")
print("   https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/available_pretrained_models.md")


In [ ]:
def run_nnunet_inference(dataset_id, configuration="2d", input_folder=None, output_folder=None,
                         folds="0", trainer="nnUNetTrainer"):
    """
    Run nnU-Net inference on new images.

    Args:
        dataset_id (int): Dataset ID
        configuration (str): Model configuration used for training
        input_folder (str): Path to folder containing test images
        output_folder (str): Path to save segmentation results
        folds (str): Folds to use for prediction (e.g., "0", "0,1,2,3,4")
        trainer (str): Trainer class name
    """

    # Set default paths if not provided
    if input_folder is None:
        input_folder = dataset_path / "imagesTs"
    if output_folder is None:
        output_folder = Path(nnunet_results) / "predictions" / f"Dataset{dataset_id:03d}"

    # Create output directory
    Path(output_folder).mkdir(parents=True, exist_ok=True)

    print(f"🎯 Running inference...")
    print(f"   Input folder: {input_folder}")
    print(f"   Output folder: {output_folder}")

    # Check if input folder exists and has images
    if not Path(input_folder).exists():
        print(f"❌ Input folder not found: {input_folder}")
        return False

    input_files = list(Path(input_folder).glob("*.nii.gz"))
    if not input_files:
        print(f"❌ No .nii.gz files found in {input_folder}")
        return False

    print(f"📁 Found {len(input_files)} images to process")

    # Build inference command
    cmd = [
        "nnUNetv2_predict",
        "-i", str(input_folder),
        "-o", str(output_folder),
        "-d", str(dataset_id),
        "-c", configuration,
        "-f", folds,
        "-tr", trainer
    ]

    print(f"🚀 Running command: {' '.join(cmd)}")

    try:
        # For demo purposes, we'll show the command but not actually run it
        # Uncomment the lines below to actually run inference

        print("💡 Inference command ready to execute.")
        print("💡 To run actual inference, uncomment the subprocess.run() lines below.")

        # result = subprocess.run(
        #     cmd,
        #     capture_output=True,
        #     text=True,
        #     timeout=3600
        # )
        #
        # if result.returncode == 0:
        #     print(f"✅ Inference completed successfully!")
        #     print(f"📁 Results saved to: {output_folder}")
        #     return True
        # else:
        #     print(f"❌ Inference failed with return code: {result.returncode}")
        #     print(f"Error: {result.stderr}")
        #     return False

        # For demo, create a dummy result
        return True

    except Exception as e:
        print(f"❌ Error during inference: {e}")
        return False

# Run inference
if training_success:
    inference_success = run_nnunet_inference(DATASET_ID, configuration="2d")
else:
    print("❌ Cannot run inference - training not completed")
    inference_success = False


In [ ]:
def load_nifti_image(filepath):
    """Load a NIfTI image and return the data array."""
    try:
        img = nib.load(filepath)
        return img.get_fdata()
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

def visualize_segmentation_results(image_path, label_path=None, prediction_path=None,
                                 slice_idx=None, figsize=(15, 5)):
    """
    Visualize medical image segmentation results.

    Args:
        image_path (str): Path to original image
        label_path (str): Path to ground truth label (optional)
        prediction_path (str): Path to prediction (optional)
        slice_idx (int): Slice index for 3D images (middle slice if None)
        figsize (tuple): Figure size
    """

    # Load images
    image_data = load_nifti_image(image_path)
    if image_data is None:
        return

    label_data = None
    if label_path and Path(label_path).exists():
        label_data = load_nifti_image(label_path)

    prediction_data = None
    if prediction_path and Path(prediction_path).exists():
        prediction_data = load_nifti_image(prediction_path)

    # Handle 3D images - select middle slice if not specified
    if len(image_data.shape) == 3:
        if slice_idx is None:
            slice_idx = image_data.shape[2] // 2
        image_slice = image_data[:, :, slice_idx]
        label_slice = label_data[:, :, slice_idx] if label_data is not None else None
        pred_slice = prediction_data[:, :, slice_idx] if prediction_data is not None else None
    else:
        image_slice = image_data
        label_slice = label_data
        pred_slice = prediction_data

    # Determine number of subplots
    n_plots = 1  # Original image
    if label_slice is not None:
        n_plots += 1
    if pred_slice is not None:
        n_plots += 1

    # Create visualization
    fig, axes = plt.subplots(1, n_plots, figsize=figsize)
    if n_plots == 1:
        axes = [axes]

    plot_idx = 0

    # Plot original image
    axes[plot_idx].imshow(image_slice.T, cmap='gray', origin='lower')
    axes[plot_idx].set_title('Original Image')
    axes[plot_idx].axis('off')
    plot_idx += 1

    # Plot ground truth if available
    if label_slice is not None:
        axes[plot_idx].imshow(image_slice.T, cmap='gray', origin='lower', alpha=0.7)
        axes[plot_idx].imshow(label_slice.T, cmap='jet', origin='lower', alpha=0.5)
        axes[plot_idx].set_title('Ground Truth Overlay')
        axes[plot_idx].axis('off')
        plot_idx += 1

    # Plot prediction if available
    if pred_slice is not None:
        axes[plot_idx].imshow(image_slice.T, cmap='gray', origin='lower', alpha=0.7)
        axes[plot_idx].imshow(pred_slice.T, cmap='jet', origin='lower', alpha=0.5)
        axes[plot_idx].set_title('Prediction Overlay')
        axes[plot_idx].axis('off')

    plt.tight_layout()
    plt.show()

    # Print some statistics
    print(f"📊 Image Statistics:")
    print(f"   Image shape: {image_data.shape}")
    print(f"   Image range: [{image_data.min():.2f}, {image_data.max():.2f}]")

    if label_data is not None:
        unique_labels = np.unique(label_data)
        print(f"   Ground truth labels: {unique_labels}")

    if prediction_data is not None:
        unique_preds = np.unique(prediction_data)
        print(f"   Prediction labels: {unique_preds}")

# Demo visualization function
def demo_visualization():
    """
    Demo visualization with synthetic data.
    """
    print("📸 Creating demo visualization...")

    # Create synthetic medical image data
    img_size = (256, 256)

    # Synthetic anatomical image (brain-like)
    x, y = np.meshgrid(np.linspace(-1, 1, img_size[0]), np.linspace(-1, 1, img_size[1]))
    r = np.sqrt(x**2 + y**2)

    # Create synthetic image with different tissue intensities
    synthetic_image = np.zeros(img_size)
    synthetic_image[r < 0.8] = 100  # Brain tissue
    synthetic_image[r < 0.6] = 150  # Gray matter
    synthetic_image[r < 0.4] = 200  # White matter
    synthetic_image += np.random.normal(0, 10, img_size)  # Add noise

    # Create synthetic segmentation
    synthetic_label = np.zeros(img_size)
    synthetic_label[r < 0.6] = 1  # Gray matter
    synthetic_label[r < 0.4] = 2  # White matter

    # Create synthetic prediction (with some errors)
    synthetic_pred = synthetic_label.copy()
    # Add some prediction errors
    noise_mask = np.random.random(img_size) < 0.05
    synthetic_pred[noise_mask] = np.random.randint(0, 3, np.sum(noise_mask))

    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(synthetic_image, cmap='gray')
    axes[0].set_title('Synthetic Medical Image')
    axes[0].axis('off')

    axes[1].imshow(synthetic_image, cmap='gray', alpha=0.7)
    axes[1].imshow(synthetic_label, cmap='jet', alpha=0.5)
    axes[1].set_title('Ground Truth Overlay')
    axes[1].axis('off')

    axes[2].imshow(synthetic_image, cmap='gray', alpha=0.7)
    axes[2].imshow(synthetic_pred, cmap='jet', alpha=0.5)
    axes[2].set_title('Prediction Overlay')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    print("✅ Demo visualization complete!")

# Run visualization
if inference_success:
    # Try to find actual files to visualize
    images_path = dataset_path / "imagesTr"
    labels_path = dataset_path / "labelsTr"
    predictions_path = Path(nnunet_results) / "predictions" / f"Dataset{DATASET_ID:03d}"

    if images_path.exists():
        image_files = list(images_path.glob("*.nii.gz"))
        if image_files:
            sample_image = image_files[0]
            sample_label = labels_path / sample_image.name if labels_path.exists() else None
            sample_pred = predictions_path / sample_image.name if predictions_path.exists() else None

            print(f"🖼️ Visualizing sample: {sample_image.name}")
            visualize_segmentation_results(
                image_path=sample_image,
                label_path=sample_label,
                prediction_path=sample_pred
            )
        else:
            print("📸 No sample images found, showing demo visualization instead")
            demo_visualization()
    else:
        print("📸 No images found, showing demo visualization instead")
        demo_visualization()
else:
    print("📸 Showing demo visualization")
    demo_visualization()
